# Data Deduplication Lab - Getting Started

Welcome to the Data Deduplication Lab. This notebook prepares your **Cloudera AI Workbench** session for the Phase 1 exercises.

## Learning Objectives

By the end of this notebook, you will:
- Install project Python dependencies from `requirements.txt`
- Create a **local-mode** Spark session in Cloudera AI Workbench (`local[*]`)
- Read sample customer data from the project `data/` directory
- Copy that local file to HDFS, then read it back
- Inspect duplicate patterns on `name` + `email`

## What this notebook does

1. **Dependencies** — install packages from `../requirements.txt`
2. **Spark setup** — create a local Spark session with an explicit HDFS nameservice (`hdfs://ns1`)
3. **Local read** — load `../data/redundant_data.csv`
4. **HDFS copy + read** — write the local data to HDFS, then verify by reading it
5. **Duplicate check** — profile uniqueness on key columns

## Prerequisites

- Cloudera AI Workbench session with PySpark available
- Project files including `use-case-phase-1/data/`
- Permission to read HDFS `/tmp` (or edit `HDFS_INPUT` to a path you can read)
- Basic Python familiarity

## Types of Deduplication (lab overview)

### Record-Level Deduplication (Exercise 1+)
- Removes duplicate **rows/records** within a dataset
- Example: two customer rows with the same name and email

**Next notebook after this setup:** `01_Basic_Deduplication.ipynb`


## 0. Install Project Requirements

Install packages from `use-case-phase-1/requirements.txt` into this session (includes `pyspark` if it is not already on the kernel path). Re-run this cell after restarting the kernel if imports fail.


In [ ]:
from pathlib import Path
import importlib
import sys
import subprocess

REQ_FILE = Path("../requirements.txt").resolve()
assert REQ_FILE.is_file(), f"Missing requirements file: {REQ_FILE}"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)])

# Ensure a fresh import after install (handles kernels that started without pyspark)
importlib.invalidate_caches()
import pyspark

print(f"✓ Installed requirements from: {REQ_FILE}")
print(f"✓ pyspark {pyspark.__version__} available")


## 1. Create Local Spark Session

This lab uses **local mode** (`local[*]`). Pip-installed PySpark often lacks cluster `core-site.xml`, so we set `fs.defaultFS` explicitly to the HA nameservice (`hdfs://ns1`). Paths must include that host/nameservice — `hdfs:///tmp/...` fails with "Incomplete HDFS URI, no host".


In [ ]:
from pyspark.sql import SparkSession

# HA nameservice from cluster hdfs-site / core-site (edit if yours differs)
HDFS_NAMESERVICE = "ns1"
HDFS_DEFAULT_FS = f"hdfs://{HDFS_NAMESERVICE}"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DeduplicationLab_GettingStarted")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.defaultFS", HDFS_DEFAULT_FS)
    .getOrCreate()
)

# Shared HDFS destination for this lab (directory of part-*.csv after Spark write)
HDFS_PATH = f"{HDFS_DEFAULT_FS}/tmp/cdp_user_demo/phase1/redundant_data.csv"

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print(f"Default FS:    {spark.sparkContext._jsc.hadoopConfiguration().get('fs.defaultFS')}")
print(f"HDFS path:     {HDFS_PATH}")
print("✓ Spark session created successfully")


## 2. Read Local Sample Data

Load the project CSV from `use-case-phase-1/data/`. Schema: `id`, `name`, `email`, `address`.

For a larger workload later, switch the filename to `redundant_data_large.csv`.


In [ ]:
from pathlib import Path

LOCAL_DATA_DIR = Path("../data").resolve()
LOCAL_INPUT = str(LOCAL_DATA_DIR / "redundant_data.csv")
# Optional scale-up file:
# LOCAL_INPUT = str(LOCAL_DATA_DIR / "redundant_data_large.csv")

df_local = spark.read.csv(LOCAL_INPUT, header=True, inferSchema=True)

print(f"✓ Loaded local file: {LOCAL_INPUT}")
print(f"Total records: {df_local.count():,}")
print(f"Columns: {', '.join(df_local.columns)}")
print("\nPreview:")
df_local.show(10, truncate=False)
df_local.printSchema()


## 3. Copy Local Data to HDFS

Write the local DataFrame to HDFS under `/tmp`, then read it back to confirm connectivity.

Spark CSV output is a **directory** containing `part-*.csv` files — that is expected.


In [ ]:
# Copy local sample data to HDFS (overwrite if present)
(
    df_local.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(HDFS_PATH)
)

print(f"✓ Wrote local data to HDFS: {HDFS_PATH}")


## 4. Read the File Back from HDFS

Confirm the copy by loading `HDFS_PATH` with Spark.


In [ ]:
df_hdfs = spark.read.csv(HDFS_PATH, header=True, inferSchema=True)

print(f"✓ Loaded HDFS file: {HDFS_PATH}")
print(f"Total records: {df_hdfs.count():,}")
print(f"Columns: {', '.join(df_hdfs.columns)}")
print("\nPreview:")
df_hdfs.show(5, truncate=False)
df_hdfs.printSchema()


## 5. Quick Duplicate Check (Local Data)

Profile uniqueness on `name` + `email` in the local dataset — the same keys used in Exercise 1.


In [ ]:
KEY_COLS = ["name", "email"]

total_count = df_local.count()
unique_count = df_local.select(*KEY_COLS).distinct().count()
duplicates_count = total_count - unique_count
duplicate_rate = (duplicates_count / total_count * 100) if total_count else 0

print(f"Total records: {total_count:,}")
print(f"Unique records (by name+email): {unique_count:,}")
print(f"Duplicate records: {duplicates_count:,}")
print(f"Duplicate rate: {duplicate_rate:.2f}%")
print(f"\nLocal path for Exercise 1:\n  {LOCAL_INPUT}")
print(f"HDFS path verified:\n  {HDFS_PATH}")


## Next Steps

Setup is complete. Exercise 1 defaults to the **local** CSV:

```text
../data/redundant_data.csv
```

HDFS copy written/verified above (edit `HDFS_NAMESERVICE` if needed):

```text
hdfs://ns1/tmp/cdp_user_demo/phase1/redundant_data.csv
```

1. **Exercise 1**: Basic Deduplication — `01_Basic_Deduplication.ipynb`
2. **Exercise 2**: Iceberg REST Catalog — `02_Iceberg_REST_Catalog.ipynb`

## Cleanup

Stop the Spark session when you are finished with this notebook. Later notebooks create their own sessions.


In [ ]:
spark.stop()
print("✓ Spark session stopped")
